# Week 6 · Class 1 — Your First Machine Learning Model 🏠📈

**90 min · hands-on** · tools: `pandas`, `matplotlib`, `seaborn`, `scikit-learn`

This notebook is the practical side of today's slides. We train our **first ML model**: a **linear regression** that predicts house prices.

| Part | Time | What we do |
|---|---|---|
| 0 | 3' | Setup |
| 1 | 10' | Warm-up: salary vs experience (a model that works perfectly) |
| 2 | 10' | Meet the California housing data |
| 3 | 12' | Quick exploration with ML in mind |
| 4 | 8' | Features, target and the train/test split |
| 5 | 5' | A baseline: the "dumb" model to beat |
| 6 | 12' | Simple regression: income → house value |
| 7 | 12' | Multiple regression: all features |
| 8 | 10' | Diagnose the errors |
| 9 | 3' | Recap |

🧑‍💻 = we write it **together**. 📚 **At home:** Part B at the end (5 exercises with solutions).

> 🧪 **Saturday labs:** *Regression Analysis* and *Intro to Scikit-Learn* use exactly the steps of this notebook.

---
## 0 · Setup ⚙️

📁 **The data**
- **Colab:** nothing to do. Colab ships the file in `sample_data/california_housing_train.csv`.
- **VS Code / Jupyter:** put `california_housing_train.csv` (or the smaller `california_housing_test.csv` from the Week 6 folder) next to this notebook.

The cell below tries each place in turn and tells you which file it found.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

sns.set_theme(style="whitegrid")
pd.set_option("display.float_format", "{:,.2f}".format)

---
## 1 · Warm-up: salary vs years of experience 💼

Before real data, a tiny clean example: **30 employees**, their years of experience and their salary.
The question: *can we predict salary from experience?*

In [ ]:
years = [1.1, 1.3, 1.5, 2.0, 2.2, 2.9, 3.0, 3.2, 3.2, 3.7, 3.9, 4.0, 4.0, 4.1, 4.5,
         4.9, 5.1, 5.3, 5.9, 6.0, 6.8, 7.1, 7.9, 8.2, 8.7, 9.0, 9.5, 9.6, 10.3, 10.5]
salary = [39343, 46205, 37731, 43525, 39891, 56642, 60150, 54445, 64445, 57189,
          63218, 55794, 56957, 57081, 61111, 67938, 66029, 83088, 81363, 93940,
          91738, 98273, 101302, 113812, 109431, 105582, 116969, 112635, 122391, 121872]
salaries = pd.DataFrame({"years": years, "salary": salary})
salaries.head()

🧑‍💻 **Code with me:** plot salary against years. Does a straight line look reasonable?

In [ ]:
plt.figure(figsize=(6, 4))
plt.scatter(salaries["years"], salaries["salary"])
plt.xlabel("years of experience")
plt.ylabel("salary (€)")
plt.title("Salary vs experience")
plt.show()

Now our first model, in three lines. scikit-learn always wants:
- **X**: a *table* of features (2D, so double brackets `[["years"]]`)
- **y**: one *column*, the target

In [ ]:
X_small = salaries[["years"]]      # 2D: a DataFrame with one column
y_small = salaries["salary"]       # 1D: a Series

model_small = LinearRegression()   # 1. create
model_small.fit(X_small, y_small)  # 2. fit = learn b0 and b1

print(f"intercept b0 = {model_small.intercept_:,.0f} €")
print(f"slope     b1 = {model_small.coef_[0]:,.0f} € per extra year")

📖 **Read it:** every extra year of experience goes with about **+9,450 €** of salary, on average. The intercept is the predicted salary at 0 years.

🧑‍💻 **Code with me:** predict the salary for 5 and 12 years, and draw the line.

In [ ]:
new = pd.DataFrame({"years": [5, 12]})
print(model_small.predict(new))     # 3. predict

plt.figure(figsize=(6, 4))
plt.scatter(salaries["years"], salaries["salary"], label="data")
grid = pd.DataFrame({"years": np.linspace(0, 12, 50)})
plt.plot(grid["years"], model_small.predict(grid), color="red", label="fitted line")
plt.xlabel("years of experience"); plt.ylabel("salary (€)"); plt.legend()
plt.show()

print("R² =", round(r2_score(y_small, model_small.predict(X_small)), 3))

✅ R² ≈ 0.96: the line explains almost all the variation. Great, but **real data is rarely this clean**, and 12 years is already outside our data (extrapolation!).
Now the real thing.

---
## 2 · Meet the California housing data 🏠

- 1990 US census, **one row = one district** (a block of a few hundred homes), not one house.
- **Target:** `median_house_value`, in dollars.
- **Features:** location, age of the houses, counts of rooms / people / households, and median income (in **tens of thousands** of $).

In [ ]:
candidates = [
    "california_housing_train.csv",
    "sample_data/california_housing_train.csv",
    "/content/sample_data/california_housing_train.csv",
    "california_housing_test.csv",
    "sample_data/california_housing_test.csv",
]
path = next((p for p in candidates if os.path.exists(p)), None)
if path is None:
    raise FileNotFoundError("Put california_housing_train.csv next to this notebook (see Setup).")

df = pd.read_csv(path)
print("Loaded:", path, "→", df.shape)
df.head()

🧑‍💻 **Code with me:** column types, missing values and a summary.

In [ ]:
df.info()

In [ ]:
df.describe().T

🤔 **Look at the max of `median_house_value`.** It is exactly 500,001. In the census, every value above 500k was written as 500,001: the target is **capped**.

In [ ]:
capped = (df["median_house_value"] >= 500_000).mean()
print(f"{capped:.1%} of districts are capped at 500k")

---
## 3 · Quick exploration with ML in mind 🔍

Before modelling we ask three questions:
1. What does the **target** look like?
2. Which features move **together with** the target?
3. Any surprises (outliers, caps, strange scales)?

🧑‍💻 **Code with me:** histogram of the target.

In [ ]:
plt.figure(figsize=(7, 4))
sns.histplot(df["median_house_value"], bins=50)
plt.title("Median house value per district")
plt.show()

The spike on the right is the cap. Skewed target, long right tail: keep this in mind for Thursday (log transforms).

🧑‍💻 **Code with me:** correlation of every feature with the target, sorted.

In [ ]:
corr = df.corr()["median_house_value"].drop("median_house_value").sort_values(ascending=False)
corr

In [ ]:
plt.figure(figsize=(6, 4))
corr.plot(kind="barh")
plt.axvline(0, color="black", lw=1)
plt.title("Correlation with median house value")
plt.show()

📖 **Income** is by far the strongest single signal. Counts like `total_rooms` barely correlate: a district with many rooms may simply be a **big** district, not a rich one.

🧑‍💻 **Code with me:** where are the expensive districts?

In [ ]:
plt.figure(figsize=(6, 5))
plt.scatter(df["longitude"], df["latitude"], c=df["median_house_value"],
            cmap="viridis", s=4, alpha=0.6)
plt.colorbar(label="median house value ($)")
plt.xlabel("longitude"); plt.ylabel("latitude")
plt.title("California districts")
plt.show()

The coast (San Francisco Bay, Los Angeles) is expensive. Location matters, but **not in a straight line**: keep that in mind when we read the results.

---
## 4 · Features, target and the train/test split ✂️

- **X** = everything except the target. **y** = the target.
- We hide **20%** of the districts in a **test set**. The model never sees them during training: they are our "new exam".
- `random_state=42` makes the random shuffle the same every time (so we all get the same numbers).

In [ ]:
X = df.drop(columns="median_house_value")
y = df["median_house_value"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("train:", X_train.shape, " test:", X_test.shape)

> ⚠️ From now on: **fit on train, evaluate on test.** Never let the test set influence a choice.

---
## 5 · A baseline: the "dumb" model to beat 🪨

The simplest possible prediction: **always predict the average price of the training set.**
Any real model must do better than this, or it is useless.

We'll measure everything with a small helper function (remember functions from Week 1?):

In [ ]:
def evaluate(y_true, y_pred, name="model"):
    # print MAE, RMSE (in $) and R² for one set of predictions
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    print(f"{name:<22} MAE = {mae:>9,.0f} $   RMSE = {rmse:>9,.0f} $   R² = {r2:6.3f}")
    return {"model": name, "MAE": mae, "RMSE": rmse, "R2": r2}

In [ ]:
baseline_pred = np.full(len(y_test), y_train.mean())
results = [evaluate(y_test, baseline_pred, "baseline (mean)")]

📖 R² = 0 (or just below): by definition, the mean explains nothing. MAE ≈ how wrong we are **on average** when we know nothing.

---
## 6 · Simple regression: income → house value 📈

One feature, exactly like the salary warm-up.

🧑‍💻 **Code with me:** fit a model on `median_income` only.

In [ ]:
model_income = LinearRegression()
model_income.fit(X_train[["median_income"]], y_train)

print(f"intercept b0 = {model_income.intercept_:,.0f} $")
print(f"slope     b1 = {model_income.coef_[0]:,.0f} $ per +1 income unit (= +10k$ of income)")

📖 **Read it:** a district whose median income is 10k$ higher has, on average, houses worth roughly **40k$ more**.

In [ ]:
pred_income = model_income.predict(X_test[["median_income"]])
results.append(evaluate(y_test, pred_income, "income only"))

In [ ]:
plt.figure(figsize=(7, 4))
plt.scatter(X_test["median_income"], y_test, s=5, alpha=0.4, label="test districts")
grid = pd.DataFrame({"median_income": np.linspace(0, 15, 50)})
plt.plot(grid["median_income"], model_income.predict(grid), color="red", lw=3, label="fitted line")
plt.xlabel("median income (tens of k$)"); plt.ylabel("median house value ($)")
plt.legend(); plt.show()

🤔 **Predict:** what will happen to R² if we give the model **all eight** features?

---
## 7 · Multiple regression: all features 🧮

Same three lines, just a bigger X. The model now learns **one coefficient per feature**.

🧑‍💻 **Code with me:**

In [ ]:
model_all = LinearRegression()
model_all.fit(X_train, y_train)

pred_all = model_all.predict(X_test)
results.append(evaluate(y_test, pred_all, "all 8 features"))

Let's put all results side by side:

In [ ]:
pd.DataFrame(results).set_index("model")

🧑‍💻 **Code with me:** is the model **overfitting**? Compare the score on the training data with the score on the test data.

In [ ]:
_ = evaluate(y_train, model_all.predict(X_train), "all 8 — TRAIN")
_ = evaluate(y_test,  pred_all,                   "all 8 — TEST")

📖 Train and test scores are close → **no overfitting**. A straight-line model is simple; its problem is more likely **underfitting** (too simple to capture everything).

Now the coefficients:

In [ ]:
coefs = pd.Series(model_all.coef_, index=X.columns).sort_values()
coefs

⚠️ **Careful when reading these:**
- **Units differ.** `median_income` is in tens of k$, `total_rooms` is a count in the thousands. A big coefficient is not automatically an important feature.
- **Correlated features.** `total_rooms`, `total_bedrooms`, `households` and `population` move together, so their coefficients can take surprising signs.
- Thursday we **scale** the features and use **regularisation**, which makes coefficients much easier to compare.

---
## 8 · Diagnose the errors 🩺

One number hides a lot. Two plots every regression deserves:
1. **Predicted vs actual:** a perfect model puts every dot on the diagonal.
2. **Residuals vs predicted:** residual = actual − predicted. We want a shapeless cloud around 0.

🧑‍💻 **Code with me:**

In [ ]:
residuals = y_test - pred_all

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].scatter(y_test, pred_all, s=5, alpha=0.4)
axes[0].plot([0, 500_000], [0, 500_000], color="red", ls="--")
axes[0].set_xlabel("actual ($)"); axes[0].set_ylabel("predicted ($)")
axes[0].set_title("Predicted vs actual")

axes[1].scatter(pred_all, residuals, s=5, alpha=0.4)
axes[1].axhline(0, color="red", ls="--")
axes[1].set_xlabel("predicted ($)"); axes[1].set_ylabel("residual ($)")
axes[1].set_title("Residuals vs predicted")

plt.tight_layout(); plt.show()

📖 **What we see:**
- A **vertical wall at 500k**: the capped districts. The model predicts above and below, but the truth is stuck at 500k.
- A **diagonal stripe** in the residual plot: that is the cap again.
- Some **negative predictions**: a straight line happily goes below 0, but prices cannot.

In [ ]:
print("negative predictions:", (pred_all < 0).sum())
print("worst miss: $", f"{residuals.abs().max():,.0f}")

🧭 **So the model is not bad, it is a baseline.** A straight line explains about 60% of the variation in prices. To do better we need:
- **better features** (rooms *per household*, income on a log scale…) → **Thursday**
- **non-linear models** (trees, boosting) → **Week 8**

---
## 9 · Recap 🧭

| Step | Code |
|---|---|
| features & target | `X = df.drop(columns="target")`, `y = df["target"]` |
| split | `train_test_split(X, y, test_size=0.2, random_state=42)` |
| create → fit → predict | `model = LinearRegression()` · `model.fit(X_train, y_train)` · `model.predict(X_test)` |
| read the model | `model.intercept_`, `model.coef_` |
| metrics | `mean_absolute_error`, `mean_squared_error` (→ √ for RMSE), `r2_score` |
| always | compare with a **baseline**, compare **train vs test**, **plot the residuals** |

---
# 📚 Part B · At home — 5 exercises

Each exercise has a hint and a solution cell. Try first, then compare.

### Exercise 1 · Predict a new district 🏡
A district at longitude −122.2, latitude 37.8, houses 30 years old, 2,000 rooms, 400 bedrooms, 1,000 people, 380 households, median income 6.0.
What does `model_all` predict?

*Hint:* build a one-row DataFrame with **the same columns, in the same order** as `X`.

In [ ]:
new_district = pd.DataFrame([{
    "longitude": -122.2, "latitude": 37.8, "housing_median_age": 30,
    "total_rooms": 2000, "total_bedrooms": 400, "population": 1000,
    "households": 380, "median_income": 6.0,
}])[X.columns]
print(f"predicted median house value: ${model_all.predict(new_district)[0]:,.0f}")

### Exercise 2 · Which single feature is the best predictor? 🥇
For **each** feature, fit a one-feature linear regression on train and compute R² on test. Show the results sorted.

*Hint:* a `for` loop over `X.columns`, and a dictionary to collect the scores.

In [ ]:
scores = {}
for col in X.columns:
    m = LinearRegression().fit(X_train[[col]], y_train)
    scores[col] = r2_score(y_test, m.predict(X_test[[col]]))
pd.Series(scores).sort_values(ascending=False)

Income wins by far. Note how weak latitude and longitude are **on their own**: location matters, but not along a straight line.

### Exercise 3 · How stable is the score? 🎲
Repeat the split + fit + R² for `random_state` = 0, 1, …, 9. What are the min, max and mean test R²?

*Hint:* put the three steps inside a loop and store each R² in a list.

In [ ]:
r2s = []
for seed in range(10):
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=seed)
    m = LinearRegression().fit(Xtr, ytr)
    r2s.append(r2_score(yte, m.predict(Xte)))
print(f"min {min(r2s):.3f}   max {max(r2s):.3f}   mean {np.mean(r2s):.3f}")

The score moves a little with the split. That is why on Thursday we use **cross-validation**: average over several splits instead of trusting one.

### Exercise 4 · What if we drop the capped districts? ✂️
Remove the rows with `median_house_value >= 500_000`, then split, fit and evaluate again. Does R² go up or down? Why might this be **dangerous** in a real product?

*Hint:* filter `df` with a boolean mask first.

In [ ]:
df_nocap = df[df["median_house_value"] < 500_000]
Xn = df_nocap.drop(columns="median_house_value"); yn = df_nocap["median_house_value"]
Xn_tr, Xn_te, yn_tr, yn_te = train_test_split(Xn, yn, test_size=0.2, random_state=42)
m_nocap = LinearRegression().fit(Xn_tr, yn_tr)
_ = evaluate(yn_te, m_nocap.predict(Xn_te), "no capped districts")

R² goes **down** (fewer extreme prices to explain, so the mean baseline is also better), and the model now **knows nothing about expensive districts**. If you deploy it, it will be badly wrong on exactly those cases. Changing the data to make a score look good is a classic trap.

### Exercise 5 · Your first engineered feature 🛠️
Create `rooms_per_household = total_rooms / households` and `income_squared = median_income ** 2`. Add them to X, refit, and compare test R² with `model_all`.

*Hint:* `df.assign(...)` or `df["new"] = ...`, then repeat the split with the same `random_state`.

In [ ]:
df_fe = df.assign(
    rooms_per_household=df["total_rooms"] / df["households"],
    income_squared=df["median_income"] ** 2,
)
Xf = df_fe.drop(columns="median_house_value"); yf = df_fe["median_house_value"]
Xf_tr, Xf_te, yf_tr, yf_te = train_test_split(Xf, yf, test_size=0.2, random_state=42)
m_fe = LinearRegression().fit(Xf_tr, yf_tr)
_ = evaluate(y_test, pred_all, "all 8 features")
_ = evaluate(yf_te, m_fe.predict(Xf_te), "+ 2 new features")

Only a small gain: a straight line still cannot bend around the non-linear patterns. Thursday we do **feature engineering** properly (scaling, logs, polynomials, pipelines) and see much bigger effects. 🚀